# LaminoRAG Demo

Personal laminopathy research assistant. Forced cross-corpus retrieval over the user's reading.

## Setup

Add repo root to `sys.path` so notebook can import sibling modules. Verify config + env vars.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

import config
print("Corpora:", config.ALL_CORPORA)
print("Embedding backend:", config.EMBEDDING_BACKEND)
print("Chroma path:", config.CHROMA_PATH)
print("Ingest model:", config.INGEST_MODEL)
print("Synthesis model:", config.SYNTH_MODEL)
print("Synthesis backend:", config.SYNTH_BACKEND)
print("LLM_API_BASE set:", bool(os.environ.get("LLM_API_BASE")))
print("LLM_API_KEY set:", bool(os.environ.get("LLM_API_KEY")))


## Ingest

Drop articles into `data/articles/<corpus>/` (e.g. `data/articles/laminopathy/foo.pdf`). The parent folder name determines the corpus. `sync()` reconciles the vector store with disk: ingests anything new, and removes chunks for files that were deleted or moved between corpus folders.

In [ ]:
import ingest

# sync() is incremental + idempotent: it skips already-ingested files,
# ingests new ones, and removes chunks whose source PDF no longer exists
# at the recorded path (handles deletes and corpus-folder moves).
result = ingest.sync(ROOT / "data" / "articles")
print(f"sync: {result}")

## Query - laminopathy-centered

In [ ]:
import query

response = query.ask(
    "Read the connections between changed gene expressions in laminopathic hearts "
    "and propose an experiment to find commonality between them."
)
print("ANSWER:\n")
print(response["answer"])
print("\nSOURCES BY CORPUS:")
for corpus, titles in response["sources"].items():
    print(f"  {corpus}: {titles}")

## Query - cross-corpus

In [ ]:
response = query.ask("Can bioinformatics approaches improve LNP targeting for laminopathy?")
print("ANSWER:\n")
print(response["answer"])
print("\nSOURCES BY CORPUS:")
for corpus, titles in response["sources"].items():
    print(f"  {corpus}: {titles}")